# Relevant Python 3.7 and 3.8 Changes
## Advanced Tutorial Problems with Complete Solutions — Alternative Edition

This notebook covers the same selected Python 3.7 and Python 3.8 changes as the
provided lesson, but all problems and examples are new.

The teaching style deliberately follows the source notebook:

- short conceptual explanations;
- one idea per cell;
- larger problems divided into logical steps;
- safe demonstrations of expected errors;
- complete solutions followed by verification.

## Topics

1. Positional-only parameters
2. Self-documenting f-strings
3. `as_integer_ratio()`
4. `lru_cache`
5. `math.dist`
6. `namedtuple` changes
7. Reversed dictionary views
8. `continue` in `finally`
9. `is` with literals
10. Cross-topic capstone exercises

## Imports

Run this cell before executing any individual exercise.

In [1]:
import math
import sys
import warnings

from collections import namedtuple
from decimal import Decimal
from fractions import Fraction
from functools import lru_cache, wraps
from inspect import Parameter, signature
from itertools import accumulate
from numbers import Integral

## Runtime check

The notebook requires Python 3.8 or newer because it uses positional-only
parameters and self-documenting f-strings.

In [2]:
print(sys.version)
assert sys.version_info >= (3, 8)

3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


# 1. Positional-Only Parameters

Parameters before `/` can only be supplied positionally. Parameters after `*`
can only be supplied by keyword.

## Problem 1 — Build a unit-conversion API

The calculation is:

```text
converted = value * factor + offset
```

Requirements:

- `value` and `factor` are positional-only;
- `offset` may be positional or keyword;
- `precision` is keyword-only;
- `precision=None` disables rounding.

### Step 1 — Design the signature

In [3]:
def convert_measurement(value, factor, /, offset=0, *, precision=None):
    pass

print(signature(convert_measurement))

(value, factor, /, offset=0, *, precision=None)


The signature now communicates the intended calling convention, but the function
still needs an implementation.

### Step 2 — Implement the calculation

In [4]:
def convert_measurement(value, factor, /, offset=0, *, precision=None):
    result = value * factor + offset

    if precision is None:
        return result

    return round(result, precision)

In [5]:
assert convert_measurement(10, 1.8, 32, precision=1) == 50.0
assert convert_measurement(10, 1.8, offset=32) == 50.0

### Step 3 — Add validation

Boolean values are instances of `int`, so they must be rejected explicitly when
they are not meaningful.

In [6]:
def convert_measurement(value, factor, /, offset=0, *, precision=None):
    if precision is not None:
        if isinstance(precision, bool) or not isinstance(precision, Integral):
            raise TypeError("precision must be an integer or None")
        if precision < 0:
            raise ValueError("precision must be non-negative")

    result = value * factor + offset

    if precision is None:
        return result

    return round(result, precision)

### Step 4 — Verify expected failures

In [7]:
try:
    convert_measurement(value=10, factor=1.8)
except TypeError as ex:
    print("Expected positional-only error:")
    print(ex)

try:
    convert_measurement(10, 1.8, 32, 2)
except TypeError as ex:
    print("\nExpected keyword-only error:")
    print(ex)

try:
    convert_measurement(10, 1.8, precision=True)
except TypeError as ex:
    print("\nExpected validation error:")
    print(ex)

Expected positional-only error:
convert_measurement() got some positional-only arguments passed as keyword arguments: 'value, factor'

Expected keyword-only error:
convert_measurement() takes from 2 to 3 positional arguments but 4 were given

Expected validation error:
precision must be an integer or None


## Problem 2 — Reuse a positional-only name inside `**metadata`

A positional-only parameter name may also appear as a key in `**kwargs`.

In [8]:
def create_event(category, payload, /, **metadata):
    return {
        "category": category,
        "payload": payload,
        "metadata": metadata,
    }

The positional argument and the metadata field have different roles even though
they use the same spelling.

In [9]:
event = create_event(
    "purchase",
    {"amount": Decimal("19.95")},
    category="financial",
    source="mobile",
)

event

{'category': 'purchase',
 'payload': {'amount': Decimal('19.95')},
 'metadata': {'category': 'financial', 'source': 'mobile'}}

In [10]:
assert event["category"] == "purchase"
assert event["metadata"]["category"] == "financial"

## Problem 3 — Inspect parameter kinds

In [11]:
def request(path, method, /, body=None, *, timeout=5, retries=2, **headers):
    return path, method, body, timeout, retries, headers

request_signature = signature(request)
print(request_signature)

(path, method, /, body=None, *, timeout=5, retries=2, **headers)


Build a dictionary showing the classification of each parameter.

In [12]:
parameter_kinds = {
    name: parameter.kind
    for name, parameter in request_signature.parameters.items()
}

parameter_kinds

{'path': <_ParameterKind.POSITIONAL_ONLY: 0>,
 'method': <_ParameterKind.POSITIONAL_ONLY: 0>,
 'body': <_ParameterKind.POSITIONAL_OR_KEYWORD: 1>,
 'timeout': <_ParameterKind.KEYWORD_ONLY: 3>,
 'retries': <_ParameterKind.KEYWORD_ONLY: 3>,
 'headers': <_ParameterKind.VAR_KEYWORD: 4>}

In [13]:
assert parameter_kinds["path"] is Parameter.POSITIONAL_ONLY
assert parameter_kinds["method"] is Parameter.POSITIONAL_ONLY
assert parameter_kinds["body"] is Parameter.POSITIONAL_OR_KEYWORD
assert parameter_kinds["timeout"] is Parameter.KEYWORD_ONLY
assert parameter_kinds["headers"] is Parameter.VAR_KEYWORD

## Problem 4 — Decorate a constrained API

`functools.wraps` preserves access to the wrapped function's original signature.

In [14]:
def report_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        print(f"  {args=}")
        print(f"  {kwargs=}")
        result = func(*args, **kwargs)
        print(f"  {result=}")
        return result

    return wrapper

In [15]:
@report_call
def calculate_area(width, height, /, *, unit="cm"):
    if width < 0 or height < 0:
        raise ValueError("dimensions must be non-negative")

    return {
        "area": width * height,
        "unit": f"{unit}²",
    }

In [16]:
print(signature(calculate_area))

area = calculate_area(4, 7, unit="m")
assert area == {"area": 28, "unit": "m²"}

(width, height, /, *, unit='cm')
Calling calculate_area
  args=(4, 7)
  kwargs={'unit': 'm'}
  result={'area': 28, 'unit': 'm²'}


In [17]:
try:
    calculate_area(width=4, height=7)
except TypeError as ex:
    print("Expected failure:")
    print(ex)

Calling calculate_area
  args=()
  kwargs={'width': 4, 'height': 7}
Expected failure:
calculate_area() got some positional-only arguments passed as keyword arguments: 'width, height'


# 2. Self-Documenting f-Strings

Python 3.8 introduced the debugging form `f"{expression=}"`.

## Problem 5 — Build a progressive diagnostic message

In [18]:
response_times = [0.12, 0.18, 0.09, 0.31, 0.20]

count = len(response_times)
total = sum(response_times)
mean = total / count
maximum = max(response_times)

First inspect the raw representations.

In [19]:
print(f"{count=}")
print(f"{total=}")
print(f"{mean=}")
print(f"{maximum=}")

count=5
total=0.9
mean=0.18
maximum=0.31


Now add format specifications for a compact human-readable diagnostic.

In [20]:
message = (
    f"{count=}, "
    f"{total=:.3f}s, "
    f"{mean=:.3f}s, "
    f"{maximum=:.3f}s"
)

print(message)

count=5, total=0.900s, mean=0.180s, maximum=0.310s


In [21]:
assert "count=5" in message
assert "mean=0.180s" in message

## Problem 6 — Observe whitespace preservation

In [22]:
x = 7
y = 11

compact = f"{x+y=}"
spaced = f"{x + y = }"

print(compact)
print(spaced)

assert compact == "x+y=18"
assert spaced == "x + y = 18"

x+y=18
x + y = 18


## Problem 7 — Compare `repr` and `str`

In [23]:
class Account:
    def __init__(self, account_id, owner):
        self.account_id = account_id
        self.owner = owner

    def __repr__(self):
        return (
            f"Account(account_id={self.account_id!r}, "
            f"owner={self.owner!r})"
        )

    def __str__(self):
        return f"{self.owner} ({self.account_id})"

In [24]:
account = Account("A-104", "Grace")

default_output = f"{account=}"
repr_output = f"{account=!r}"
str_output = f"{account=!s}"

print(default_output)
print(repr_output)
print(str_output)

account=Account(account_id='A-104', owner='Grace')
account=Account(account_id='A-104', owner='Grace')
account=Grace (A-104)


In [25]:
assert default_output == repr_output
assert str_output == "account=Grace (A-104)"

## Problem 8 — Avoid repeated side effects

In [26]:
ticket_counter = 0

def issue_ticket():
    global ticket_counter
    ticket_counter += 1
    return f"T-{ticket_counter:04d}"

Writing the call twice evaluates it twice.

In [27]:
ticket_counter = 0
unsafe_message = f"{issue_ticket()=}, {issue_ticket()=}"

print(unsafe_message)
print(f"{ticket_counter=}")

assert ticket_counter == 2

issue_ticket()='T-0001', issue_ticket()='T-0002'
ticket_counter=2


Evaluate once, store the result, and debug the variable.

In [28]:
ticket_counter = 0

ticket = issue_ticket()
safe_message = f"{ticket=}"

print(safe_message)
print(f"{ticket_counter=}")

assert ticket_counter == 1

ticket='T-0001'
ticket_counter=1


# 3. `as_integer_ratio()` and Numeric Duck Typing

Python 3.8 added `as_integer_ratio()` to `int`, `bool`, and `Fraction`.
`float` and `Decimal` already supported the same protocol.

## Problem 9 — Convert different numeric types exactly

In [29]:
values = [
    8,
    Fraction(5, 12),
    Decimal("0.125"),
    0.5,
    True,
]

for value in values:
    print(type(value).__name__, value.as_integer_ratio())

int (8, 1)
Fraction (5, 12)
Decimal (1, 8)
float (1, 2)
bool (1, 1)


`True` behaves numerically like `1`. In many applications, however, accepting a
boolean as a measurement is undesirable.

### Step 1 — Implement a protocol-based converter

In [30]:
def to_exact_fraction(value, /, *, allow_bool=False):
    if isinstance(value, bool) and not allow_bool:
        raise TypeError("boolean values are not accepted")

    ratio_method = getattr(value, "as_integer_ratio", None)

    if ratio_method is None:
        raise TypeError(
            f"{type(value).__name__} does not support as_integer_ratio()"
        )

    numerator, denominator = ratio_method()

    if denominator <= 0:
        raise ValueError("the denominator must be positive")

    return Fraction(numerator, denominator)

### Step 2 — Verify several built-in numeric types

In [31]:
assert to_exact_fraction(8) == Fraction(8, 1)
assert to_exact_fraction(Fraction(5, 12)) == Fraction(5, 12)
assert to_exact_fraction(Decimal("0.125")) == Fraction(1, 8)
assert to_exact_fraction(0.5) == Fraction(1, 2)

try:
    to_exact_fraction(True)
except TypeError as ex:
    print("Expected boolean rejection:")
    print(ex)

Expected boolean rejection:
boolean values are not accepted


## Problem 10 — Inspect the exact value stored by a float

In [32]:
float_ratio = to_exact_fraction(0.1)
decimal_ratio = to_exact_fraction(Decimal("0.1"))

print(f"{float_ratio=}")
print(f"{decimal_ratio=}")

float_ratio=Fraction(3602879701896397, 36028797018963968)
decimal_ratio=Fraction(1, 10)


The `Decimal` value is exactly one tenth. The float is a nearby rational number.

In [33]:
difference = float_ratio - decimal_ratio

print(f"{difference=}")
print(f"{float(difference)=:.22e}")

assert decimal_ratio == Fraction(1, 10)
assert difference != 0

difference=Fraction(1, 180143985094819840)
float(difference)=5.5511151231257830102669e-18


## Problem 11 — Create a compatible custom numeric object

Duck typing allows a custom object to participate by implementing the expected
method.

In [34]:
class Percentage:
    def __init__(self, numerator, denominator=100):
        fraction = Fraction(numerator, denominator)
        self._numerator = fraction.numerator
        self._denominator = fraction.denominator

    def as_integer_ratio(self):
        return self._numerator, self._denominator

    def __repr__(self):
        return (
            f"Percentage({self._numerator}, "
            f"{self._denominator})"
        )

In [35]:
discount = Percentage(12, 100)
exact_discount = to_exact_fraction(discount)

print(discount)
print(exact_discount)

assert exact_discount == Fraction(3, 25)

Percentage(3, 25)
3/25


The converter did not need a special branch for `Percentage`.

## Problem 12 — Normalize mixed numeric weights

The function will:

1. convert every input exactly;
2. reject negative weights;
3. reject an empty or all-zero collection;
4. return exact normalized fractions.

In [36]:
def normalize_exact_weights(values):
    exact_values = [to_exact_fraction(value) for value in values]

    if not exact_values:
        raise ValueError("at least one weight is required")

    if any(value < 0 for value in exact_values):
        raise ValueError("weights must be non-negative")

    total = sum(exact_values, start=Fraction(0, 1))

    if total == 0:
        raise ValueError("at least one weight must be positive")

    normalized = [value / total for value in exact_values]

    assert sum(normalized, start=Fraction(0, 1)) == 1
    return normalized

In [37]:
raw_weights = [
    2,
    Decimal("0.5"),
    Fraction(1, 4),
    Percentage(25),
]

normalized_weights = normalize_exact_weights(raw_weights)

print(normalized_weights)

assert normalized_weights == [
    Fraction(2, 3),
    Fraction(1, 6),
    Fraction(1, 12),
    Fraction(1, 12),
]

[Fraction(2, 3), Fraction(1, 6), Fraction(1, 12), Fraction(1, 12)]


# 4. `lru_cache`

Python 3.8 allows `@lru_cache` without empty parentheses when no arguments are
needed.

## Problem 13 — Count paths through a blocked grid

A robot starts at `(0, 0)` and may move only right or down. Some cells are
blocked. The recursive state consists only of the current coordinates.

In [38]:
blocked_cells = frozenset({
    (1, 1),
    (2, 3),
})

target_row = 4
target_column = 5

Use a cached recursive function so repeated coordinates are solved once.

In [39]:
@lru_cache
def count_paths(row, column):
    if row > target_row or column > target_column:
        return 0

    if (row, column) in blocked_cells:
        return 0

    if (row, column) == (target_row, target_column):
        return 1

    return (
        count_paths(row + 1, column)
        + count_paths(row, column + 1)
    )

In [40]:
path_count = count_paths(0, 0)
cache_statistics = count_paths.cache_info()

print(f"{path_count=}")
print(f"{cache_statistics=}")

assert path_count > 0
assert cache_statistics.hits > 0

path_count=32
cache_statistics=CacheInfo(hits=16, misses=39, maxsize=128, currsize=39)


## Problem 14 — Observe LRU eviction

In [41]:
calculation_count = 0

@lru_cache(maxsize=3)
def expensive_double(value):
    global calculation_count
    calculation_count += 1
    print(f"calculating {value}")
    return value * 2

Fill the cache with keys `1`, `2`, and `3`.

In [42]:
calculation_count = 0
expensive_double.cache_clear()

for value in [1, 2, 3]:
    expensive_double(value)

print(expensive_double.cache_info())

calculating 1
calculating 2
calculating 3
CacheInfo(hits=0, misses=3, maxsize=3, currsize=3)


Access key `1` again. This is a hit and refreshes its recency.

In [43]:
assert expensive_double(1) == 2
print(expensive_double.cache_info())

CacheInfo(hits=1, misses=3, maxsize=3, currsize=3)


Add key `4`. The least recently used key is now `2`, so it is evicted.

In [44]:
assert expensive_double(4) == 8
print(expensive_double.cache_info())

calculating 4
CacheInfo(hits=1, misses=4, maxsize=3, currsize=3)


Requesting key `2` again should create another miss.

In [45]:
before = expensive_double.cache_info()
assert expensive_double(2) == 4
after = expensive_double.cache_info()

print(f"{before=}")
print(f"{after=}")

assert after.misses == before.misses + 1

calculating 2
before=CacheInfo(hits=1, misses=4, maxsize=3, currsize=3)
after=CacheInfo(hits=1, misses=5, maxsize=3, currsize=3)


## Problem 15 — Cache a function whose public input is a list

Lists are unhashable. Convert the public list to an immutable tuple before
calling the cached internal function.

In [46]:
@lru_cache
def _prefix_totals(values):
    return tuple(accumulate(values))

def prefix_totals(values, /):
    return _prefix_totals(tuple(values))

In [47]:
_prefix_totals.cache_clear()

first = prefix_totals([3, 1, 4, 1])
second = prefix_totals([3, 1, 4, 1])

print(first)
print(_prefix_totals.cache_info())

assert first == (3, 4, 8, 9)
assert second == first
assert _prefix_totals.cache_info().hits == 1

(3, 4, 8, 9)
CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)


## Problem 16 — Clear cache state between tests

In [48]:
@lru_cache
def power_of_two(exponent):
    return 2 ** exponent

In [49]:
power_of_two.cache_clear()

assert power_of_two(10) == 1024
assert power_of_two(10) == 1024
assert power_of_two.cache_info().hits == 1

power_of_two.cache_clear()

empty_info = power_of_two.cache_info()
print(empty_info)

assert empty_info.hits == 0
assert empty_info.misses == 0
assert empty_info.currsize == 0

CacheInfo(hits=0, misses=0, maxsize=128, currsize=0)


## Problem 17 — Use type-sensitive cache keys

In [50]:
typed_call_count = 0

@lru_cache(maxsize=None, typed=True)
def identify_numeric_type(value):
    global typed_call_count
    typed_call_count += 1
    return type(value).__name__

In [51]:
typed_call_count = 0
identify_numeric_type.cache_clear()

type_names = [
    identify_numeric_type(1),
    identify_numeric_type(1.0),
    identify_numeric_type(True),
    identify_numeric_type(Decimal("1")),
]

print(type_names)
print(identify_numeric_type.cache_info())

assert type_names == ["int", "float", "bool", "Decimal"]
assert typed_call_count == 4

['int', 'float', 'bool', 'Decimal']
CacheInfo(hits=0, misses=4, maxsize=None, currsize=4)


# 5. `math.dist`

`math.dist(p, q)` calculates Euclidean distance in any number of dimensions.

## Problem 18 — Calculate the length of a polyline

In [52]:
polyline = [
    (0, 0),
    (3, 4),
    (6, 4),
    (6, 12),
]

segments = list(zip(polyline, polyline[1:]))
segments

[((0, 0), (3, 4)), ((3, 4), (6, 4)), ((6, 4), (6, 12))]

Calculate each segment separately.

In [53]:
segment_lengths = [
    math.dist(start, end)
    for start, end in segments
]

segment_lengths

[5.0, 3.0, 8.0]

Now build a validated reusable function.

In [54]:
def polyline_length(points, /):
    points = [tuple(point) for point in points]

    if len(points) < 2:
        return 0.0

    dimensions = {len(point) for point in points}

    if len(dimensions) != 1:
        raise ValueError("all points must have the same dimension")

    return sum(
        math.dist(start, end)
        for start, end in zip(points, points[1:])
    )

In [55]:
length = polyline_length(polyline)

print(length)

assert length == 16.0
assert polyline_length([(0, 0, 0)]) == 0.0

16.0


## Problem 19 — Assign points to their nearest center

In [56]:
centers = {
    "north": (0, 10),
    "south": (0, -10),
    "east": (10, 0),
    "west": (-10, 0),
}

points = [
    (1, 7),
    (8, 2),
    (-7, -1),
    (0, -8),
]

In [57]:
def nearest_center(point, centers, /):
    point = tuple(point)

    if not centers:
        raise ValueError("at least one center is required")

    if any(len(center) != len(point) for center in centers.values()):
        raise ValueError("center dimensions must match the point")

    name = min(
        centers,
        key=lambda candidate_name: math.dist(
            point,
            centers[candidate_name],
        ),
    )

    return name, math.dist(point, centers[name])

In [58]:
assignments = {
    point: nearest_center(point, centers)
    for point in points
}

assignments

{(1, 7): ('north', 3.1622776601683795),
 (8, 2): ('east', 2.8284271247461903),
 (-7, -1): ('west', 3.1622776601683795),
 (0, -8): ('south', 2.0)}

In [59]:
assert assignments[(1, 7)][0] == "north"
assert assignments[(8, 2)][0] == "east"
assert assignments[(-7, -1)][0] == "west"
assert assignments[(0, -8)][0] == "south"

## Problem 20 — Verify a five-dimensional distance

In [60]:
point_a = (1, 2, 3, 4, 5)
point_b = (6, 5, 4, 3, 2)

distance_5d = math.dist(point_a, point_b)

manual_5d = math.sqrt(
    sum(
        (left - right) ** 2
        for left, right in zip(point_a, point_b)
    )
)

print(f"{distance_5d=}")
print(f"{manual_5d=}")

assert math.isclose(distance_5d, manual_5d)

distance_5d=6.708203932499369
manual_5d=6.708203932499369


# 6. `namedtuple` Changes in Python 3.7 and 3.8

Python 3.7 added the `defaults=` argument and `_field_defaults`.
Python 3.8 changed `_asdict()` to return a regular ordered `dict`.

## Problem 21 — Apply defaults from right to left

We want a task record with two required fields and two optional fields:

```text
Task(name, owner, priority, enabled)
```

The defaults should be `"normal"` and `True`.

In [61]:
Task = namedtuple(
    "Task",
    "name owner priority enabled",
    defaults=("normal", True),
)

The two defaults apply to the two rightmost fields.

In [62]:
task_a = Task("backup", "Ada")
task_b = Task("deploy", "Grace", "high")
task_c = Task("cleanup", "Linus", enabled=False)

print(task_a)
print(task_b)
print(task_c)

Task(name='backup', owner='Ada', priority='normal', enabled=True)
Task(name='deploy', owner='Grace', priority='high', enabled=True)
Task(name='cleanup', owner='Linus', priority='normal', enabled=False)


In [63]:
assert task_a == Task("backup", "Ada", "normal", True)
assert task_b == Task("deploy", "Grace", "high", True)
assert task_c == Task("cleanup", "Linus", "normal", False)

## Problem 22 — Inspect `_field_defaults`

In [64]:
Task._field_defaults

{'priority': 'normal', 'enabled': True}

In [65]:
assert Task._field_defaults == {
    "priority": "normal",
    "enabled": True,
}

Build a complete policy that marks fields without defaults as required.

In [66]:
REQUIRED = object()

task_field_policy = {
    field: Task._field_defaults.get(field, REQUIRED)
    for field in Task._fields
}

readable_policy = {
    field: "<required>" if value is REQUIRED else value
    for field, value in task_field_policy.items()
}

readable_policy

{'name': '<required>',
 'owner': '<required>',
 'priority': 'normal',
 'enabled': True}

In [67]:
assert readable_policy == {
    "name": "<required>",
    "owner": "<required>",
    "priority": "normal",
    "enabled": True,
}

## Problem 23 — Demonstrate the mutable-default trap

A tuple is immutable, but an object stored inside it may still be mutable.

In [68]:
UnsafeMessage = namedtuple(
    "UnsafeMessage",
    "text tags",
    defaults=([],),
)

In [69]:
message_a = UnsafeMessage("first")
message_b = UnsafeMessage("second")

message_a.tags.append("urgent")

print(message_a)
print(message_b)

assert message_a.tags is message_b.tags
assert message_b.tags == ["urgent"]

UnsafeMessage(text='first', tags=['urgent'])
UnsafeMessage(text='second', tags=['urgent'])


Both instances share the same default list.

Replace the default with `None` and create a fresh list in a factory.

In [70]:
Message = namedtuple(
    "Message",
    "text tags",
    defaults=(None,),
)

def make_message(text, tags=None):
    return Message(
        text=text,
        tags=[] if tags is None else list(tags),
    )

In [71]:
safe_a = make_message("first")
safe_b = make_message("second")

safe_a.tags.append("urgent")

print(safe_a)
print(safe_b)

assert safe_a.tags is not safe_b.tags
assert safe_b.tags == []

Message(text='first', tags=['urgent'])
Message(text='second', tags=[])


## Problem 24 — Convert a record with `_asdict()`

In [72]:
task_data = task_a._asdict()

print(type(task_data))
print(task_data)

assert type(task_data) is dict
assert list(task_data) == [
    "name",
    "owner",
    "priority",
    "enabled",
]

<class 'dict'>
{'name': 'backup', 'owner': 'Ada', 'priority': 'normal', 'enabled': True}


Because the result is a regular dictionary, it can be enriched normally.

In [73]:
task_data["summary"] = (
    f"{task_a.name} owned by {task_a.owner}"
)

task_data

{'name': 'backup',
 'owner': 'Ada',
 'priority': 'normal',
 'enabled': True,
 'summary': 'backup owned by Ada'}

In [74]:
assert list(task_data)[-1] == "summary"

## Problem 25 — Add validation through a factory

In [75]:
Coordinate = namedtuple(
    "Coordinate",
    "latitude longitude",
)

def make_coordinate(latitude, longitude):
    if isinstance(latitude, bool) or isinstance(longitude, bool):
        raise TypeError("coordinates must not be boolean")

    if not -90 <= latitude <= 90:
        raise ValueError("latitude must be between -90 and 90")

    if not -180 <= longitude <= 180:
        raise ValueError("longitude must be between -180 and 180")

    return Coordinate(latitude, longitude)

In [76]:
sofia = make_coordinate(42.6977, 23.3219)

print(sofia)

try:
    make_coordinate(120, 23)
except ValueError as ex:
    print("Expected validation failure:")
    print(ex)

Coordinate(latitude=42.6977, longitude=23.3219)
Expected validation failure:
latitude must be between -90 and 90


# 7. Reversing Dictionary Views

Python 3.8 allows `reversed()` to work with dictionaries and their key, value,
and item views.

## Problem 26 — Implement an undo operation

In [77]:
change_log = {
    "change-001": ("theme", "light", "dark"),
    "change-002": ("language", "en", "bg"),
    "change-003": ("notifications", True, False),
}

Inspect the most recently inserted item.

In [78]:
last_change_id, last_change = next(
    iter(reversed(change_log.items()))
)

print(last_change_id)
print(last_change)

change-003
('notifications', True, False)


Now build a reusable function. The last key is captured before the dictionary is
mutated.

In [79]:
def undo_latest(change_log, settings, /):
    if not change_log:
        raise LookupError("there is no change to undo")

    latest_key = next(iter(reversed(change_log)))
    field, old_value, new_value = change_log.pop(latest_key)

    if settings.get(field) != new_value:
        raise ValueError(
            "current setting does not match the recorded change"
        )

    settings[field] = old_value

    return latest_key, field, old_value

In [80]:
settings = {
    "theme": "dark",
    "language": "bg",
    "notifications": False,
}

undone = undo_latest(change_log, settings)

print(f"{undone=}")
print(f"{settings=}")
print(f"{change_log=}")

assert undone == ("change-003", "notifications", True)
assert settings["notifications"] is True

undone=('change-003', 'notifications', True)
settings={'theme': 'dark', 'language': 'bg', 'notifications': True}
change_log={'change-001': ('theme', 'light', 'dark'), 'change-002': ('language', 'en', 'bg')}


## Problem 27 — Find the latest matching record

In [81]:
status_history = {
    "10:00": "queued",
    "10:04": "running",
    "10:07": "warning",
    "10:12": "running",
    "10:18": "complete",
}

In [82]:
def latest_matching_item(mapping, predicate, /):
    for key, value in reversed(mapping.items()):
        if predicate(key, value):
            return key, value

    raise LookupError("no matching item")

In [83]:
latest_active = latest_matching_item(
    status_history,
    lambda timestamp, status: status in {"running", "warning"},
)

print(latest_active)

assert latest_active == ("10:12", "running")

('10:12', 'running')


## Problem 28 — Mutate safely after taking a snapshot

Dictionary views are live. When deletion is required, collect the relevant keys
before modifying the dictionary.

In [84]:
def remove_latest_entries(mapping, count, /):
    if count < 0:
        raise ValueError("count must be non-negative")

    keys_to_remove = list(reversed(mapping))[:count]
    removed = []

    for key in keys_to_remove:
        removed.append((key, mapping.pop(key)))

    return removed

In [85]:
queue = {
    "job-a": "done",
    "job-b": "failed",
    "job-c": "done",
    "job-d": "queued",
}

removed = remove_latest_entries(queue, 2)

print(removed)
print(queue)

assert removed == [
    ("job-d", "queued"),
    ("job-c", "done"),
]
assert queue == {
    "job-a": "done",
    "job-b": "failed",
}

[('job-d', 'queued'), ('job-c', 'done')]
{'job-a': 'done', 'job-b': 'failed'}


# 8. `continue` Inside `finally`

Python 3.8 permits `continue` in a `finally` block. The syntax is legal, but it
can suppress exceptions or override earlier control flow.

## Problem 29 — Observe exception suppression

In [86]:
def unsafe_filter(values):
    accepted = []
    cleanup_log = []

    for value in values:
        try:
            if value < 0:
                raise ValueError(f"negative value: {value}")

            accepted.append(value)

        finally:
            cleanup_log.append(f"cleaned {value}")

            if value < 0:
                continue

    return accepted, cleanup_log

In [87]:
accepted, cleanup_log = unsafe_filter([2, -5, 7])

print(f"{accepted=}")
print(f"{cleanup_log=}")

assert accepted == [2, 7]
assert len(cleanup_log) == 3

accepted=[2, 7]
cleanup_log=['cleaned 2', 'cleaned -5', 'cleaned 7']


The exception did not reach the caller because `continue` replaced the pending
exceptional control flow.

## Problem 30 — Rewrite the control flow safely

In [88]:
def safe_filter(values):
    accepted = []
    rejected = []
    cleanup_log = []

    for value in values:
        try:
            if value < 0:
                rejected.append((value, "negative"))
                continue

            accepted.append(value)

        finally:
            cleanup_log.append(f"cleaned {value}")

    return accepted, rejected, cleanup_log

In [89]:
accepted, rejected, cleanup_log = safe_filter([2, -5, 7])

print(f"{accepted=}")
print(f"{rejected=}")
print(f"{cleanup_log=}")

assert accepted == [2, 7]
assert rejected == [(-5, "negative")]
assert len(cleanup_log) == 3

accepted=[2, 7]
rejected=[(-5, 'negative')]
cleanup_log=['cleaned 2', 'cleaned -5', 'cleaned 7']


## Problem 31 — Use a context manager for cleanup

In [90]:
class LoggedResource:
    def __init__(self, name, log):
        self.name = name
        self.log = log

    def __enter__(self):
        self.log.append(f"open:{self.name}")
        return self

    def __exit__(self, exc_type, exc, traceback):
        self.log.append(f"close:{self.name}")
        return False

In [91]:
resource_log = []
processed = []

for value in [3, -1, 5]:
    with LoggedResource(str(value), resource_log):
        if value < 0:
            continue

        processed.append(value * 10)

print(processed)
print(resource_log)

assert processed == [30, 50]
assert resource_log == [
    "open:3",
    "close:3",
    "open:-1",
    "close:-1",
    "open:5",
    "close:5",
]

[30, 50]
['open:3', 'close:3', 'open:-1', 'close:-1', 'open:5', 'close:5']


# 9. Identity, Equality, and `SyntaxWarning`

`is` checks identity. `==` checks value equality. Python 3.8 warns about
comparisons such as `value is 1`.

## Problem 32 — Capture the warning safely

In [92]:
warning_source = '''
value = 1
result = value is 1
'''

with warnings.catch_warnings(record=True) as captured_warnings:
    warnings.simplefilter("always", SyntaxWarning)

    compile(
        warning_source,
        "<identity-example>",
        "exec",
    )

warning_messages = [
    str(item.message)
    for item in captured_warnings
]

warning_messages

['"is" with \'int\' literal. Did you mean "=="?']

In [93]:
assert any(
    "Did you mean" in message
    and "==" in message
    for message in warning_messages
)

## Problem 33 — Compare equal but distinct objects

In [94]:
left = [1, 2, 3]
right = [1, 2, 3]
alias = left

print(f"{left == right=}")
print(f"{left is right=}")
print(f"{left is alias=}")

assert left == right
assert left is not right
assert left is alias

left == right=True
left is right=False
left is alias=True


## Problem 34 — Use identity for a private sentinel

In [95]:
NOT_SUPPLIED = object()

def read_value(mapping, key, /, *, fallback=NOT_SUPPLIED):
    value = mapping.get(key, NOT_SUPPLIED)

    if value is NOT_SUPPLIED:
        if fallback is NOT_SUPPLIED:
            raise KeyError(key)

        return fallback

    return value

In [96]:
configuration = {
    "timeout": None,
    "retries": 0,
}

assert read_value(configuration, "timeout") is None
assert read_value(configuration, "retries") == 0
assert read_value(
    configuration,
    "missing",
    fallback="default",
) == "default"

try:
    read_value(configuration, "missing")
except KeyError as ex:
    print("Expected missing-key error:")
    print(ex)

Expected missing-key error:
'missing'


# 10. Capstone Project — A Cached Sensor-Route Analyzer

This project combines the selected Python 3.7 and 3.8 features into one guided
workflow.

The analyzer will:

- store readings in a `namedtuple`;
- use right-aligned defaults;
- normalize mixed numeric weights exactly;
- calculate route length with `math.dist`;
- cache repeated route calculations;
- constrain the public API with `/` and `*`;
- use self-documenting f-strings;
- convert records with `_asdict()`;
- search recent reports using reversed dictionary views.

## Step 1 — Define the reading record

Only `reading_id`, `position`, and `value` are required.

The optional fields default to:

- `weight=1`
- `status="ok"`

In [97]:
Reading = namedtuple(
    "Reading",
    "reading_id position value weight status",
    defaults=(1, "ok"),
)

## Step 2 — Create a validated factory

In [98]:
def make_reading(
    reading_id,
    position,
    value,
    weight=1,
    status="ok",
):
    position = tuple(position)

    if not position:
        raise ValueError("position must contain coordinates")

    exact_weight = to_exact_fraction(weight)

    if exact_weight < 0:
        raise ValueError("weight must be non-negative")

    return Reading(
        reading_id=reading_id,
        position=position,
        value=value,
        weight=exact_weight,
        status=status,
    )

## Step 3 — Create sample readings

In [99]:
readings = (
    make_reading(
        "R-001",
        (0, 0),
        Decimal("12.5"),
    ),
    make_reading(
        "R-002",
        (3, 4),
        Decimal("15.0"),
        Percentage(50),
        "warning",
    ),
    make_reading(
        "R-003",
        (6, 8),
        Decimal("14.0"),
        Fraction(3, 2),
    ),
    make_reading(
        "R-004",
        (6, 12),
        Decimal("18.0"),
        2,
        "warning",
    ),
)

readings

(Reading(reading_id='R-001', position=(0, 0), value=Decimal('12.5'), weight=Fraction(1, 1), status='ok'),
 Reading(reading_id='R-002', position=(3, 4), value=Decimal('15.0'), weight=Fraction(1, 2), status='warning'),
 Reading(reading_id='R-003', position=(6, 8), value=Decimal('14.0'), weight=Fraction(3, 2), status='ok'),
 Reading(reading_id='R-004', position=(6, 12), value=Decimal('18.0'), weight=Fraction(2, 1), status='warning'))

## Step 4 — Cache route length

A tuple of coordinate tuples is immutable and hashable, so it is suitable as a
cache key.

In [100]:
@lru_cache
def cached_route_length(route):
    if len(route) < 2:
        return 0.0

    dimensions = {len(point) for point in route}

    if len(dimensions) != 1:
        raise ValueError(
            "all route points must have equal dimensions"
        )

    return sum(
        math.dist(start, end)
        for start, end in zip(route, route[1:])
    )

## Step 5 — Build the analyzer API

The readings are positional-only. Filtering options are keyword-only.

In [101]:
def analyze_readings(
    readings,
    /,
    *,
    include_status=None,
    limit=None,
):
    readings = tuple(readings)

    if limit is not None:
        if isinstance(limit, bool) or not isinstance(limit, Integral):
            raise TypeError("limit must be an integer or None")
        if limit < 0:
            raise ValueError("limit must be non-negative")

    selected = [
        reading
        for reading in readings
        if (
            include_status is None
            or reading.status == include_status
        )
    ]

    if limit is not None:
        selected = selected[:limit]

    route = tuple(reading.position for reading in selected)
    route_length = cached_route_length(route)

    rows = []

    for reading in selected:
        row = reading._asdict()

        row["weighted_value"] = (
            Decimal(reading.value)
            * Decimal(reading.weight.numerator)
            / Decimal(reading.weight.denominator)
        )

        rows.append(row)

    print(
        f"{include_status=}, "
        f"{limit=}, "
        f"{len(selected)=}, "
        f"{route_length=:.3f}"
    )

    return {
        "rows": rows,
        "route_length": route_length,
    }

## Step 6 — Analyze warning readings

In [102]:
warning_report = analyze_readings(
    readings,
    include_status="warning",
    limit=2,
)

warning_report

include_status='warning', limit=2, len(selected)=2, route_length=8.544


{'rows': [{'reading_id': 'R-002',
   'position': (3, 4),
   'value': Decimal('15.0'),
   'weight': Fraction(1, 2),
   'status': 'warning',
   'weighted_value': Decimal('7.5')},
  {'reading_id': 'R-004',
   'position': (6, 12),
   'value': Decimal('18.0'),
   'weight': Fraction(2, 1),
   'status': 'warning',
   'weighted_value': Decimal('36.0')}],
 'route_length': 8.54400374531753}

In [103]:
assert [
    row["reading_id"]
    for row in warning_report["rows"]
] == ["R-002", "R-004"]

assert math.isclose(
    warning_report["route_length"],
    math.dist((3, 4), (6, 12)),
)

## Step 7 — Confirm cache reuse

In [104]:
before = cached_route_length.cache_info()

warning_report_again = analyze_readings(
    readings,
    include_status="warning",
    limit=2,
)

after = cached_route_length.cache_info()

print(f"{before=}")
print(f"{after=}")

assert after.hits == before.hits + 1

include_status='warning', limit=2, len(selected)=2, route_length=8.544
before=CacheInfo(hits=0, misses=1, maxsize=128, currsize=1)
after=CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)


## Step 8 — Store reports in insertion order

In [105]:
report_history = {
    "run-001": analyze_readings(
        readings,
        limit=2,
    ),
    "run-002": analyze_readings(
        readings,
        include_status="warning",
    ),
    "run-003": analyze_readings(
        readings,
        include_status="ok",
    ),
}

include_status=None, limit=2, len(selected)=2, route_length=5.000
include_status='warning', limit=None, len(selected)=2, route_length=8.544
include_status='ok', limit=None, len(selected)=2, route_length=10.000


## Step 9 — Find the latest report containing warning data

In [106]:
latest_warning_report = latest_matching_item(
    report_history,
    lambda run_id, report: any(
        row["status"] == "warning"
        for row in report["rows"]
    ),
)

print(latest_warning_report[0])

assert latest_warning_report[0] == "run-002"

run-002


# 11. Additional Advanced Drills

These shorter exercises provide more practice while retaining complete
solutions.

## Problem 35 — A safer slicing API

Create `take(sequence, count, /, *, from_end=False)`.

In [107]:
def take(sequence, count, /, *, from_end=False):
    if isinstance(count, bool) or not isinstance(count, Integral):
        raise TypeError("count must be an integer")

    if count < 0:
        raise ValueError("count must be non-negative")

    if count == 0:
        return sequence[:0]

    if from_end:
        return sequence[-count:]

    return sequence[:count]

In [108]:
assert take([1, 2, 3, 4], 2) == [1, 2]
assert take([1, 2, 3, 4], 2, from_end=True) == [3, 4]
assert take("python", 0) == ""

## Problem 36 — A formatted matrix diagnostic

In [109]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
]

rows = len(matrix)
columns = len(matrix[0])
matrix_total = sum(sum(row) for row in matrix)

matrix_message = (
    f"{rows=}, "
    f"{columns=}, "
    f"{matrix_total=}"
)

print(matrix_message)

assert matrix_message == (
    "rows=2, columns=3, matrix_total=21"
)

rows=2, columns=3, matrix_total=21


## Problem 37 — Exact percentage allocation

In [110]:
percentages = [
    Percentage(40),
    Decimal("0.35"),
    Fraction(1, 4),
]

exact_percentages = [
    to_exact_fraction(value)
    for value in percentages
]

print(exact_percentages)

assert exact_percentages == [
    Fraction(2, 5),
    Fraction(7, 20),
    Fraction(1, 4),
]
assert sum(
    exact_percentages,
    start=Fraction(0, 1),
) == 1

[Fraction(2, 5), Fraction(7, 20), Fraction(1, 4)]


## Problem 38 — Reverse only the values

In [111]:
scores = {
    "Ada": 91,
    "Grace": 95,
    "Linus": 88,
    "Guido": 97,
}

score_iterator = reversed(scores.values())

latest_three_scores = [
    next(score_iterator)
    for _ in range(3)
]

print(latest_three_scores)

assert latest_three_scores == [97, 88, 95]

[97, 88, 95]


## Problem 39 — Build a symmetric distance table

In [112]:
named_points = {
    "A": (0, 0),
    "B": (3, 4),
    "C": (6, 8),
}

distance_table = {
    left_name: {
        right_name: math.dist(
            left_point,
            right_point,
        )
        for right_name, right_point
        in named_points.items()
    }
    for left_name, left_point
    in named_points.items()
}

distance_table

{'A': {'A': 0.0, 'B': 5.0, 'C': 10.0},
 'B': {'A': 5.0, 'B': 0.0, 'C': 5.0},
 'C': {'A': 10.0, 'B': 5.0, 'C': 0.0}}

In [113]:
assert distance_table["A"]["B"] == 5.0
assert distance_table["B"]["A"] == 5.0
assert distance_table["A"]["C"] == 10.0
assert distance_table["C"]["C"] == 0.0

## Problem 40 — Verify named-tuple field order

In [114]:
Record = namedtuple(
    "Record",
    "identifier created active",
    defaults=(True,),
)

record = Record("X-1", "2026-07-29")
record_dictionary = record._asdict()

print(record_dictionary)

assert list(record_dictionary) == [
    "identifier",
    "created",
    "active",
]
assert record_dictionary["active"] is True

{'identifier': 'X-1', 'created': '2026-07-29', 'active': True}


## Problem 41 — Build a cache-friendly text analyzer

The public function accepts any iterable of words. The internal cached function
accepts a tuple.

In [115]:
@lru_cache
def _word_profile(words):
    return {
        "count": len(words),
        "unique": len(set(words)),
        "longest": max(words, key=len) if words else None,
    }

def word_profile(words, /):
    return _word_profile(tuple(words))

In [116]:
_word_profile.cache_clear()

profile_a = word_profile(["python", "cache", "python"])
profile_b = word_profile(("python", "cache", "python"))

print(profile_a)
print(_word_profile.cache_info())

assert profile_a == {
    "count": 3,
    "unique": 2,
    "longest": "python",
}
assert profile_b == profile_a
assert _word_profile.cache_info().hits == 1

{'count': 3, 'unique': 2, 'longest': 'python'}
CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)


## Problem 42 — Use a named tuple as a point

A named tuple is still a tuple, so `math.dist` can consume it directly.

In [117]:
Point3D = namedtuple(
    "Point3D",
    "x y z",
    defaults=(0, 0, 0),
)

origin_3d = Point3D()
point_3d = Point3D(2, 3, 6)

distance = math.dist(origin_3d, point_3d)

print(distance)

assert distance == 7.0

7.0


# 12. Final Verification

In [118]:
assert convert_measurement(
    100,
    0.01,
    precision=3,
) == 1.0

assert create_event(
    "login",
    {},
    category="security",
)["metadata"]["category"] == "security"

assert to_exact_fraction(
    Percentage(12, 48)
) == Fraction(1, 4)

assert prefix_totals(
    [1, 2, 3, 4]
) == (1, 3, 6, 10)

assert polyline_length(
    [(0, 0), (3, 4)]
) == 5.0

assert Task(
    "test",
    "Ada",
).priority == "normal"

assert latest_matching_item(
    {"a": 1, "b": 4, "c": 3},
    lambda key, value: value % 2 == 0,
) == ("b", 4)

assert read_value(
    {"x": None},
    "x",
) is None

print("All final verification checks passed.")

All final verification checks passed.


# 13. Review Checklist

### Positional-only parameters

- Is the parameter name an implementation detail?
- Would keyword-only configuration make calls clearer?

### Self-documenting f-strings

- Is the expression safe to evaluate?
- Could the output expose confidential information?

### `as_integer_ratio()`

- Do you need the exact stored value?
- Should boolean values be accepted?

### `lru_cache`

- Are all keys hashable?
- Can the cached result become stale?
- Should the cache be bounded or cleared between tests?

### `math.dist`

- Do all points have equal dimensions?
- Is Euclidean distance the intended metric?

### `namedtuple`

- Are defaults aligned with the rightmost fields?
- Are any defaults mutable?
- Is factory-level validation needed?

### Reversed dictionary views

- Is reverse insertion order the desired order?
- Will the mapping be mutated during iteration?

### `finally`

- Does the block perform cleanup only?
- Could a control-transfer statement suppress an exception?

### Identity

- Are you comparing a sentinel or singleton?
- Should ordinary value equality use `==` instead?